In [1]:
from langgraph.graph import StateGraph, START,END
from pydantic import BaseModel, Field
from typing import TypedDict,Literal
from langchain_groq import ChatGroq
from dotenv import load_dotenv
import operator
load_dotenv()

True

In [2]:
class QuadraticState(TypedDict):
    a: float
    b: float
    c: float

    equation: str
    discriminant: float
    result: str

In [3]:
def calculate_equation(state: QuadraticState) :
    equation = f"{state['a']}x^2 + {state['b']}x + {state['c']} = 0"
    return {"equation": equation}

In [4]:
def calculate_discriminant(state: QuadraticState) :
    determinant = state['b'] ** 2 - 4 * state['a'] * state['c']
    return {"discriminant":determinant}

In [5]:
def real_roots(state:QuadraticState):
    root1 = (-state['b'] + (state['b'] ** 2 - 4 * state['a'] * state['c']) ** 0.5) / 2 * state['a']
    root2 = (-state['b'] - (state['b'] ** 2 - 4 * state['a'] * state['c']) ** 0.5) / 2 * state['a']
    result = f"The roots are {root1} and {root2}"

    return {"result":result}
             

In [6]:
def repeated_roots(state:QuadraticState):
    root = -state['b'] / 2 * state['a']

    result = f"The repeated root is {root}"
    return {"result":result}

In [8]:
def no_real_roots(state: QuadraticState):

    result = f"No real roots"

    return {'result': result}


In [9]:
def check_condition(state: QuadraticState) -> Literal["real_roots", "repeated_roots", "no_real_roots"]:

    if state['discriminant'] > 0:
        return "real_roots"
    elif state['discriminant'] == 0:
        return "repeated_roots"
    else:
        return "no_real_roots"

In [13]:
graph = StateGraph(QuadraticState)

graph.add_node("equation", calculate_equation)
graph.add_node("discriminant", calculate_discriminant)
graph.add_node("real_roots",real_roots)
graph.add_node("repeated_roots",repeated_roots)
graph.add_node("no_real_roots",no_real_roots)


graph.add_edge(START, "equation")
graph.add_edge("equation", "discriminant")
graph.add_conditional_edges("discriminant",check_condition)
graph.add_edge("real_roots",END)
graph.add_edge("repeated_roots",END)
graph.add_edge("no_real_roots",END)

workflow = graph.compile()




In [14]:
initial_state = {
    "a":7,
    "b":3,
    "c":2
}
result = workflow.invoke(initial_state)
print(result)

{'a': 7, 'b': 3, 'c': 2, 'equation': '7x^2 + 3x + 2 = 0', 'discriminant': -47, 'result': 'No real roots'}
